# Water body detection from satellite images


This notebook finds rivers, lagoons and ponds in Sentinel-2 satellite images, draws a water map for the dry season
and for the wet season, and shows where water appeared or disappeared.



In [ ]:
#@title Your only input: Earth Engine project ID { display-mode: "form" }
# Where to find it: console.cloud.google.com -> click the project name at the top -> copy the value in the ID column.
# It looks like  my-project-123456  (it is NOT the project's display name).
EE_PROJECT_ID = ""  #@param {type:"string"}

## Step 1 · Set up (nothing to do)

In [ ]:
#@title Set up the project files { display-mode: "form" }
import base64, io, os, pathlib, subprocess, sys, zipfile

IN_COLAB = "google.colab" in sys.modules
PACKAGE_ZIP_B64 = "..."

work = pathlib.Path("/content/waterdet_project") if IN_COLAB else pathlib.Path("waterdet_project").resolve()
if not (work / "config.yaml").exists():
    work.mkdir(parents=True, exist_ok=True)
    zipfile.ZipFile(io.BytesIO(base64.b64decode(PACKAGE_ZIP_B64))).extractall(work)
os.chdir(work)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[gee]"], check=True)
sys.path.insert(0, str(work / "src"))
print("Ready. Working folder:", work)

In [ ]:
import json, logging
from pathlib import Path
from waterdet.config import load_config, validate_config

logging.basicConfig(level=logging.INFO, format="%(message)s")
if not EE_PROJECT_ID.strip():
    raise SystemExit("Type your Earth Engine project ID in the box near the top of the notebook, then run all again.")

cfg = load_config("config.yaml")
cfg["gee"]["project"] = EE_PROJECT_ID.strip()
validate_config(cfg)
print("Study area (west, south, east, north):", cfg["aoi"]["bbox"], "- Lagos lagoon area")
for name, p in cfg["periods"].items():
    print(f"Season '{name}': {p['start']} to {p['end']}")

## Step 2 · Sign in to Earth Engine
A pop-up asks you to sign in with Google. Approve it.

In [ ]:
from waterdet import gee_export as G
from waterdet.config import require_gee_project

G.init_ee(require_gee_project(cfg))
print("Earth Engine is connected.")

## Step 3 · Check there are enough satellite images
This only looks; it decides nothing. It stops with a clear message if there are too few images to continue.

In [ ]:
from IPython.display import Image, display

counts = G.preflight(cfg)
for period, c in counts.items():
    print(period, c)
tp = cfg["training_period"]
if counts[tp]["after_scene_cloud_filter"] < 3:
    raise SystemExit(f"Only {counts[tp]['after_scene_cloud_filter']} usable satellite images for the {tp} season.")
for period in cfg["periods"]:
    print(f"--- true-colour preview: {period} ---")
    display(Image(url=G.preview_url(cfg, period)))

## Step 4 · Earth Engine prepares the images (5 to 15 minutes)
The progress prints every 30 seconds. You can leave it.

In [ ]:
import time

tasks = G.start_exports(cfg)
while True:
    status = [t.status() for t in tasks]
    states = [s["state"] for s in status]
    print(time.strftime("%H:%M:%S"), states)
    failed = [s for s in status if s["state"] in ("FAILED", "CANCELLED")]
    if failed:
        raise RuntimeError(f"Export failed: {[s.get('error_message') for s in failed]}.")
    if all(s == "COMPLETED" for s in states):
        break
    time.sleep(30)
print("Earth Engine has finished.")

## Step 5 · Copy the prepared images from your Google Drive
A second pop-up asks for Drive access. Approve it.

In [ ]:
import shutil, time
from pathlib import Path

DATA = Path("data")
DATA.mkdir(exist_ok=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    src_dir = Path("/content/drive/MyDrive") / cfg["gee"]["drive_folder"]
    for period in cfg["periods"]:
        matches = []
        for _ in range(40):                       # Drive can lag a minute or two behind the export
            matches = sorted(src_dir.glob(f"s2_{period}*.tif"), key=lambda f: f.stat().st_mtime)
            if matches:
                break
            time.sleep(15)
        if not matches:
            raise FileNotFoundError(f"No s2_{period}*.tif in {src_dir}.")
        if any("-0000" in m.name for m in matches):
            raise RuntimeError(f"The {period} export was split into tiles.")
        shutil.copy(matches[-1], DATA / f"s2_{period}.tif")      # newest file if you have run this before
        print("copied", matches[-1].name)
else:
    print("Not on Colab: put s2_<season>.tif from your Drive folder into the data/ folder.")
    print("Present now:", sorted(p.name for p in DATA.glob("s2_*.tif")))

## Step 6 · Train the model and make the maps (a few minutes)
The computer learns what water looks like, tests itself on areas it has not seen, compares itself against simple
water-index thresholds, and draws the maps.

In [ ]:
from waterdet import pipeline

summary = pipeline.run(cfg, data_dir="data", out_dir="outputs")
for n in summary["notes"]:
    print("NOTE:", n)

## Step 7 · Your results

In [ ]:
from IPython.display import Markdown, display, Image

display(Markdown(Path("outputs/summary.md").read_text(encoding="utf-8")))
mode = cfg["labels"]["primary_mode"]
for f in sorted(Path("outputs/maps").glob("*.png")) + [Path(f"outputs/mode_{mode}") / n for n in ("cv_depth_curve.png", "tree.png")]:
    if Path(f).exists():
        print(f)
        display(Image(filename=str(f)))
print(Path(f"outputs/mode_{mode}/tree_rules.txt").read_text(encoding="utf-8"))

## Step 8 · Presentation pack and app data
Makes slide-ready images (the satellite image, the data the model sees, the detection, the change map, the decision tree,
the key numbers) in `outputs/presentation/`, and the small files for the Streamlit app in `outputs/streamlit_data/`.

In [ ]:
from waterdet.presentation import build_presentation_pack

pack = build_presentation_pack(cfg, data_dir="data", out_dir="outputs")
print("Slide images:", *pack["files"]["presentation"], sep="\n  ")
for f in ("03_before_after_dry.png", "04_model_inputs_dry.png", "06_dry_wet_change.png", "07_key_numbers.png"):
    p = Path(pack["presentation_dir"]) / f
    if p.exists():
        display(Image(filename=str(p)))
shutil.make_archive("waterdet_presentation", "zip", "outputs")
if IN_COLAB:
    from google.colab import files
    files.download("waterdet_presentation.zip")

## Step 9 · Download everything

In [ ]:
shutil.make_archive("waterdet_results", "zip", "outputs")
print("Wrote waterdet_results.zip")
if IN_COLAB:
    from google.colab import files
    files.download("waterdet_results.zip")

---
## Optional, later: get a true accuracy number
You do not need this to finish the project. The numbers above measure agreement with a global water map (JRC), which
is useful but is not proof the map matches reality. If you later want a true accuracy figure, label about 200 points by
looking at satellite images. The steps are in `docs/OPTIONAL_ACCURACY_CHECK.md`. The two cells below are for that.

In [ ]:
frame = Path("outputs/reference_frame")
frozen = Path("data/reference_points.meta.json")
if frozen.exists():
    print("Frame already frozen, not overwritten:", frozen)
else:
    shutil.copy(frame / "reference_frame.meta.json", frozen)
    print("Frozen:", frozen)
print("Label the CSV from this frame:", frame / "reference_frame.csv")
if IN_COLAB:
    from google.colab import files
    files.download(str(frame / "reference_frame.csv"))
    files.download(str(frame / "reference_frame.kml"))

In [ ]:
# After labelling, upload the finished CSV, then re-run Steps 6 and 7.
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    assert len(uploaded) == 1, "Upload exactly one file"
    shutil.move(next(iter(uploaded)), "data/reference_points.csv")
print("Reference CSV present:", Path("data/reference_points.csv").exists())